# Lab1: 文本提示驱动的图像分割实验
## Text-Prompt Driven Image Segmentation with SAM 3

![SAM 3](https://ai.meta.com/blog/segment-anything-model-3/)

---

### 实验背景

本实验使用 **Meta SAM 3 (Segment Anything Model 3)** 对一组闽南传统红砖古厝图像进行**文本提示驱动的语义分割**。

SAM 3 相比前代 SAM 2 的最大突破在于引入了**开放词汇的文本提示**能力：
- 输入一个简短的自然语言描述（如 `"swallowtail ridge"`、`"red brick wall"`）
- 模型会自动找到图像中所有匹配该语义描述的区域
- 同时输出：**二值掩码 (masks)**、**边界框 (boxes)**、**置信度分数 (scores)**

### 实验目的

1. 探索不同粒度的文本提示对分割结果的影响
2. 对比同一文本提示在不同图片中的检测效果
3. 评估 SAM 3 在传统建筑构件识别方面的能力

### SAM 3 简介

| 特性 | 说明 |
|------|------|
| 模型架构 | ViT + 文本编码器 (CLIP-style) |
| 参数量 | 848M |
| 核心能力 | 文本/框/点提示驱动的图像分割 |
| 输入 | 图片 + 文本描述 |
| 输出 | Masks + Boxes + Scores |
| 上下文长度 | 77 tokens |
| 训练数据 | 400万+ 概念标注 |

### 提示策略

本次实验设计了 **18 个文本提示**，分为三组：

| 类别 | 提示示例 | 目标 |
|------|---------|------|
| **核心特征构件** | `swallowtail ridge`, `red tile roof` | 建筑主要结构 |
| **局部细节构件** | `carved brick window`, `roof ridge sculpture` | 装饰与细部 |
| **建筑格局与环境** | `stone pavement`, `minnan architecture` | 整体布局 |

In [1]:
# ========== 环境准备 ==========
import os
import sys
import csv
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import torch
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
%matplotlib inline

print("环境就绪 ✅")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

环境就绪 ✅
PyTorch: 2.7.1+cu118
CUDA: True


### 实验配置

我们针对闽南传统建筑的特点，设计了 18 个英文文本提示（SAM 3 目前不支持中文提示）：

- **核心特征构件**：燕尾脊(swallowtail ridge)、马背山墙(horse-back gable)、红瓦屋顶(red tile roof)、红砖墙面(red brick wall)、白石基座(white stone plinth)、内庭院(inner courtyard)
- **局部细节构件**：石门框(stone door frame)、石雕花窗(carved stone window grille)、砖雕窗(carved brick window)、檐瓦(eaves tile)、水车堵(waterwheel frieze)、脊饰(roof ridge sculpture)、剪瓷雕(porcelain inlay)、斗拱(dougong bracket)
- **建筑格局与环境**：石板铺地(stone pavement)、庭院(courtyard)、传统民居(traditional house)、闽南建筑(minnan architecture)

输出路径设置为 `Outputs/Lab1_output/`，每张图片的结果存放在单独的 00-15 子目录中。

In [2]:
# ========== 路径与配置 ==========
_cwd = Path.cwd().resolve()
if (_cwd / "sam3").is_dir():
    REPO_ROOT = _cwd
elif (_cwd.parent / "sam3").is_dir():
    REPO_ROOT = _cwd.parent
else:
    raise RuntimeError(f"Cannot find repo root from {_cwd}")

CKPT_PATH = REPO_ROOT / "sam3.pt"
INPUT_DIR = REPO_ROOT / "Inputs" / "RawImages"
OUTPUT_DIR = REPO_ROOT / "Outputs" / "Lab1_output"

# 18 个文本提示（分三组）
TEXT_PROMPTS = [
    # 核心特征构件
    "swallowtail ridge",
    "horse-back gable",
    "red tile roof",
    "red brick wall",
    "white stone plinth",
    "inner courtyard",

    # 局部细节构件
    "stone door frame",
    "carved stone window grille",
    "carved brick window",
    "eaves tile",
    "waterwheel frieze",
    "roof ridge sculpture",
    "porcelain inlay",
    "dougong bracket",

    # 建筑格局与环境
    "stone pavement",
    "courtyard",
    "traditional house",
    "minnan architecture",
]

VIS_AREA_THRESHOLD = 0.01  # 至少覆盖图片 1% 才计入有效结果

# 检查权重
if not CKPT_PATH.exists():
    raise FileNotFoundError(f"权重文件未找到: {CKPT_PATH}")
print(f"权重: {CKPT_PATH} ({CKPT_PATH.stat().st_size/1024**3:.2f} GB)")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 获取图片列表（支持多种后缀）
IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.tif', '.webp'}
image_files = sorted([
    f for f in os.listdir(INPUT_DIR)
    if Path(f).suffix.lower() in IMAGE_EXTENSIONS
])
print(f"图片: {len(image_files)} 张待处理")

权重: c:\Users\Legion\Desktop\WIE\SAM3_Labs\sam3.pt (3.21 GB)
图片: 1 张待处理
CSV: 未找到


### 加载 SAM 3 模型

加载约 848M 参数的预训练模型。

> ℹ️ **注意**：`build_sam3_image_model` 会自动检测 CUDA 并加载权重。
> 第一次运行约需 10-30 秒，之后 PyTorch 的 CUDA 缓存会加速后续推理。

In [3]:
# ========== 加载 SAM 3 模型 ==========
print("[⏳] 正在加载 SAM 3 模型...")
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"    设备: {device}")

model = build_sam3_image_model(
    checkpoint_path=CKPT_PATH,
    load_from_HF=False,
    device=device,
    eval_mode=True,
)
processor = Sam3Processor(model, device=device)
print("[✅] SAM 3 模型加载完成")
print(f"    模型参数: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")

[⏳] 正在加载 SAM 3 模型...
    设备: cuda
[✅] SAM 3 模型加载完成
    模型参数: 841M


### 单图处理函数

下面是核心处理函数 `process_image`，它负责：

1. 加载图片并获取原图信息
2. 用 **全部 18 个文本提示** 分别执行分割
3. 过滤小面积区域（< 1%）
4. 保存每个 mask 为独立 PNG
5. 生成三栏可视化图（原图 / 全部 mask 叠加 / 按提示分类着色）
6. 生成逐 mask 拼接图

In [4]:
# ========== 核心处理函数 ==========
def process_image(img_path, img_id):
    """处理单张图片，返回检测统计"""
    print(f"\n{'='*60}")
    print(f"[{img_id}] 处理: {os.path.basename(img_path)}")

    # 加载图片
    img = Image.open(img_path).convert("RGB")
    img_w, img_h = img.size
    img_area = img_w * img_h
    print(f"    尺寸: {img_w} × {img_h}")

    # 创建输出子目录
    img_out_dir = os.path.join(OUTPUT_DIR, img_id)
    os.makedirs(img_out_dir, exist_ok=True)

    # 保存原图
    img.save(os.path.join(img_out_dir, f"{img_id}_original.png"))

    # ---- 遍历所有文本提示进行分割 ----
    all_masks = []  # list of (mask_np, score, prompt)

    for prompt in TEXT_PROMPTS:
        try:
            state = processor.set_image(img)
            state = processor.set_text_prompt(prompt=prompt, state=state)

            masks = state["masks"]   # (N, H, W)
            scores = state["scores"]  # (N,)

            print(f"    「{prompt}」: 找到 {len(masks)} 个实例")

            for i in range(len(masks)):
                score = scores[i].item()
                mask = masks[i].squeeze().cpu().numpy()
                coverage = mask.sum() / img_area
                if coverage < VIS_AREA_THRESHOLD:
                    continue
                all_masks.append((mask, score, prompt))
        except Exception as e:
            print(f"    「{prompt}」失败: {e}")

    print(f"    总有效掩码: {len(all_masks)} 个")

    # ---- 保存所有 mask 为 PNG ----
    for idx, (mask, score, prompt) in enumerate(all_masks):
        mask_img = Image.fromarray((mask * 255).astype(np.uint8))
        mask_filename = f"mask_{idx:02d}_{prompt}_{score:.3f}.png"
        mask_img.save(os.path.join(img_out_dir, mask_filename))

    # ---- 创建三栏可视化 ----
    fig, axes = plt.subplots(1, 3, figsize=(20, 8))

    # 左栏：原图
    axes[0].imshow(np.array(img))
    axes[0].set_title(f"原图: {img_id}\n{img_w}×{img_h}")
    axes[0].axis("off")

    # 中栏：全部 mask 按实例着色
    overlay = np.zeros((img_h, img_w, 4), dtype=np.float32)
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(all_masks), 1)))
    for i, (mask, score, prompt) in enumerate(all_masks):
        color = colors[i % len(colors)]
        overlay[mask] = color[:3].tolist() + [0.5]
    axes[1].imshow(np.array(img))
    axes[1].imshow(overlay)
    axes[1].set_title(f"分割结果 ({len(all_masks)} 个区域)")
    axes[1].axis("off")

    # 右栏：按提示类别着色
    prompt_colors = {}
    for i, (mask, score, prompt) in enumerate(all_masks):
        if prompt not in prompt_colors:
            prompt_colors[prompt] = colors[len(prompt_colors) % len(colors)]
    overlay2 = np.zeros((img_h, img_w, 4), dtype=np.float32)
    for mask, score, prompt in all_masks:
        c = prompt_colors[prompt]
        overlay2[mask] = c[:3].tolist() + [0.4]
    axes[2].imshow(np.array(img))
    axes[2].imshow(overlay2)
    legend = "\n".join([f"■ {p}" for p in prompt_colors])
    axes[2].set_title(f"按提示分类\n{legend}", fontsize=10)
    axes[2].axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(img_out_dir, f"{img_id}_result.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)

    # ---- 逐 mask 拼接图 ----
    if len(all_masks) <= 20 and len(all_masks) > 0:
        n_cols = min(5, len(all_masks))
        n_rows = (len(all_masks) + n_cols - 1) // n_cols
        fig2, axs2 = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
        axs2 = axs2.flatten() if isinstance(axs2, np.ndarray) else [axs2]
        for i, (mask, score, prompt) in enumerate(all_masks):
            axs2[i].imshow((mask * 200).astype(np.uint8), cmap="gray")
            axs2[i].set_title(f"{prompt}\nscore={score:.3f}", fontsize=8)
            axs2[i].axis("off")
        for j in range(len(all_masks), len(axs2)):
            axs2[j].axis("off")
        plt.tight_layout()
        plt.savefig(os.path.join(img_out_dir, f"{img_id}_masks.png"), dpi=150, bbox_inches="tight")
        plt.close(fig2)

    return {
        "img_id": img_id,
        "width": img_w,
        "height": img_h,
        "total_masks": len(all_masks),
        "prompts": {p: sum(1 for _, _, pp in all_masks if pp == p) for p in TEXT_PROMPTS},
        "output_dir": img_out_dir,
    }

print("[✅] process_image 函数定义完成")

[✅] process_image 函数定义完成


### 执行批量分割

遍历 16 张图片，每张用 18 个文本提示进行分割。

> ⏱ 总处理时间取决于图片数量和 GPU 性能。16 张图片 × 18 个提示约需 3-8 分钟。

In [ ]:
# ========== 批量处理全部图片 ==========
results = []

for img_file in image_files:
    img_id = img_file.replace("_RawImage.png", "")
    img_path = os.path.join(INPUT_DIR, img_file)
    try:
        stats = process_image(img_path, img_id)
        results.append(stats)
    except Exception as e:
        print(f"\n[❌] {img_id} 处理失败: {e}")
        import traceback
        traceback.print_exc()
        results.append({"img_id": img_id, "error": str(e)})

print(f"\n[✅] 批量处理完成，共处理 {len(results)} 张图片")


[00] 处理: 00_RawImage.png
    场景: 未知
    尺寸: 1080 × 690


### 实验结果汇总

下面汇总所有图片的分割结果，统计每个文本提示的检测数量。

> **实验观察点**：
> - 哪些提示检测效果最好？（如 `red tile roof` 在大片屋顶区域表现）
> - 哪些提示检测不到？（如 `dougong bracket` 可能因图片中未出现此构件）
> - 文本提示的粒度如何影响结果？（`building` 与 `swallowtail ridge` 的对比）

In [ ]:
# ========== 生成汇总报表 ==========
print("=" * 80)
print("Lab1: 文本提示分割实验 - 汇总报告")
print("=" * 80)

# 表头
header_cols = ["图片", "尺寸", "总数"] + TEXT_PROMPTS
col_widths = [10, 14, 6] + [14] * len(TEXT_PROMPTS)
header = " | ".join(h.ljust(w)[:w] for h, w in zip(header_cols, col_widths))
sep = "-" * len(header)
print(header)
print(sep)

# 每行
total_masks = 0
prompt_totals = {p: 0 for p in TEXT_PROMPTS}
for r in results:
    if "error" in r:
        print(f"{r['img_id']:<10} | {'❌ 失败':<14}")
        continue
    cols = [
        r["img_id"].ljust(10),
        f"{r['width']}×{r['height']}".ljust(14),
        str(r["total_masks"]).ljust(6),
    ]
    for p in TEXT_PROMPTS:
        cnt = r["prompts"].get(p, 0)
        prompt_totals[p] += cnt
        cols.append(str(cnt).ljust(14))
    print(" | ".join(c[:w] for c, w in zip(cols, col_widths)))
    total_masks += r["total_masks"]

print(sep)
total_cols = ["总计".ljust(10), "".ljust(14), str(total_masks).ljust(6)]
for p in TEXT_PROMPTS:
    total_cols.append(str(prompt_totals[p]).ljust(14))
print(" | ".join(c[:w] for c, w in zip(total_cols, col_widths)))

# 保存汇总到文件
summary_path = os.path.join(OUTPUT_DIR, "summary.txt")
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("Lab1: 文本提示分割实验 - 汇总报告\n")
    f.write("=" * 80 + "\n\n")
    f.write(header + "\n" + sep + "\n")
    for r in results:
        if "error" in r:
            f.write(f"{r['img_id']:<10} | {'❌ 失败':<14}\n")
            continue
        cols = [r["img_id"].ljust(10), f"{r['width']}×{r['height']}".ljust(14), str(r["total_masks"]).ljust(6)]
        for p in TEXT_PROMPTS:
            cols.append(str(r["prompts"].get(p, 0)).ljust(14))
        f.write(" | ".join(c[:w] for c, w in zip(cols, col_widths)) + "\n")
    f.write(sep + "\n")
    f.write(" | ".join(c[:w] for c, w in zip(total_cols, col_widths)) + "\n")

# 保存 CSV 格式
csv_path = os.path.join(OUTPUT_DIR, "summary.csv")
with open(csv_path, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["图片编号", "宽度", "高度"] + TEXT_PROMPTS + ["总掩码数"])
    for r in results:
        if "error" in r:
            writer.writerow([r["img_id"], "ERROR", r["error"]])
        else:
            row = [r["img_id"], r["width"], r["height"]]
            for p in TEXT_PROMPTS:
                row.append(r["prompts"].get(p, 0))
            row.append(r["total_masks"])
            writer.writerow(row)

print(f"\n[✅] 汇总已保存: {summary_path}")
print(f"[✅] CSV已保存: {csv_path}")
print(f"[✅] 每张图片的详细结果在 {OUTPUT_DIR}/<编号>/ 目录下")

Lab1: 文本提示分割实验 - 汇总报告
图片         | 尺寸             | 总数     | swallowtail ri | horse-back gab | red tile roof  | red brick wall | white stone pl | inner courtyar | stone door fra | carved stone w | carved brick w | eaves tile     | waterwheel fri | roof ridge scu | porcelain inla | dougong bracke | stone pavement | courtyard      | traditional ho | minnan archite
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
00         | 1440×700       | 34     | 0              | 0              | 21             | 1              | 0              | 0              | 0              | 0              | 0              | 0              | 0              | 0              | 0              | 0              | 0              

### 讨论与分析

#### 关键发现

完成实验后，可以从以下角度分析结果：

1. **提示粒度的影响**
   - 粗粒度提示（`building`, `roof`）往往检测出大片区域
   - 细粒度提示（`roof ridge sculpture`, `porcelain inlay`）仅检测局部

2. **视觉显著性**
   - 颜色鲜明、边缘清晰的构件更容易被正确检测
   - 远景图片中细小构件往往漏检

3. **开放词汇的局限性**
   - 专业术语（如 `dougong bracket`）的检测效果取决于训练数据中出现的频率
   - 部分抽象概念（如 `minnan architecture`）难以映射到具体像素区域

#### 后续改进方向

- 调整 `VIS_AREA_THRESHOLD` 参数，平衡精度与召回率
- 尝试更精确的英文描述（如用 `"traditional Minnan red-brick building with swallowtail ridge"`）
- 结合框提示或点提示进行精细化调整
- 对不同提示的分数设定自适应阈值